[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/14_diffusion_flow_targets.ipynb)

# 14. Diffusion and flow training targets — correct target definitions

이 노트북은 모델 구조가 아니라 **학습 target이 어떻게 정의되는지**를 비교한다.

이전 MeanFlow section은 임의의 analytic field를 구간 평균낸 것뿐이라 MeanFlow training objective가 아니었다. 이번 버전은 논문의 핵심인 **average velocity identity와 JVP target**을 실제로 계산한다.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.func import jvp

torch.manual_seed(7)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


## 1. Diffusion forward noising

diffusion 계열에서는 clean sample `x0`와 Gaussian noise `epsilon`을 schedule coefficient `alpha_t`, `sigma_t`로 섞어 `x_t = alpha_t x0 + sigma_t epsilon`을 만든다.


In [ ]:
x0 = torch.tensor(
    [[1.0, -1.0]],
    device=device,
)
epsilon = torch.tensor(
    [[0.5, 2.0]],
    device=device,
)
alpha_t = torch.tensor(
    [[0.8]],
    device=device,
)
sigma_t = torch.sqrt(1 - alpha_t.square())

x_t = alpha_t * x0 + sigma_t * epsilon

print("x_t:", x_t)


## 2. epsilon, x0, and v parameterizations

같은 noisy input `x_t`를 주더라도 network가 어떤 quantity를 직접 예측하도록 학습할지 선택할 수 있다.


In [ ]:
target_epsilon = epsilon
target_x0 = x0
target_v = alpha_t * epsilon - sigma_t * x0

print("epsilon target:", target_epsilon)
print("x0 target:", target_x0)
print("v target:", target_v)


## 3. Flow Matching conditional path

가장 단순한 straight conditional path를 `z_t=(1-t)x+t epsilon`으로 잡으면 instantaneous conditional velocity는 `v=epsilon-x`가 된다. network는 이 velocity field를 회귀한다.


In [ ]:
data = torch.tensor(
    [[1.0, -1.0]],
    device=device,
)
noise = torch.tensor(
    [[-0.5, 1.5]],
    device=device,
)
t = torch.tensor(
    [0.3],
    device=device,
)

z_t = (1 - t[:, None]) * data + t[:, None] * noise
instantaneous_velocity = noise - data

print("z_t:", z_t)
print("instantaneous velocity:", instantaneous_velocity)


## 4. Rectified Flow and reflow distinction

straight interpolation target 자체는 Flow Matching에서도 쓸 수 있다. Rectified Flow의 중요한 추가 아이디어는 학습된 flow로 endpoint pair를 다시 만들고 **reflow**를 반복해 transport trajectory를 더 straight하게 만드는 것이다. 단순히 여러 `t`에서 같은 `epsilon-x`를 출력하는 것만으로 reflow를 구현한 것은 아니다.


In [ ]:
source = torch.tensor([[1.0, -1.0]], device=device)
destination = torch.tensor([[-0.5, 1.5]], device=device)

def straight_velocity(source, destination):
    return destination - source

velocity = straight_velocity(source, destination)

for t_value in [0.0, 0.25, 0.5, 0.75, 1.0]:
    t_value = torch.tensor(t_value, device=device)
    point = (1 - t_value) * source + t_value * destination

    print(
        "t=", float(t_value),
        "point=", point.tolist(),
        "velocity=", velocity.tolist(),
    )


## 5. MeanFlow: average velocity is a finite-interval quantity

MeanFlow는 instantaneous velocity `v(z_t,t)` 대신 interval `[r,t]`의 average velocity `u(z_t,r,t)`를 직접 모델링한다. 정의를 매 training step마다 적분해서 target으로 만들면 비싸므로 논문은 다음 identity를 사용한다.

`u(z_t,r,t) = v(z_t,t) - (t-r) d/dt u(z_t,r,t)`

여기서 total derivative는 `d/dt u = v·∂_z u + ∂_t u`이며 JVP로 계산한다.


In [ ]:
class TinyMeanVelocity(nn.Module):
    def __init__(self, data_dim=2, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(data_dim + 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, data_dim),
        )

    def forward(self, z, r, t):
        condition = torch.cat(
            [z, r[:, None], t[:, None]],
            dim=-1,
        )
        return self.net(condition)


mean_velocity_model = TinyMeanVelocity().to(device)

data = torch.randn(4, 2, device=device)
noise = torch.randn_like(data)

t = torch.rand(4, device=device)
r = torch.rand(4, device=device) * t

z_t = (1 - t[:, None]) * data + t[:, None] * noise
v = noise - data

def mean_velocity_function(z, r, t):
    return mean_velocity_model(z, r, t)

u_prediction, du_dt = jvp(
    mean_velocity_function,
    (z_t, r, t),
    (
        v,
        torch.zeros_like(r),
        torch.ones_like(t),
    ),
)

meanflow_target = (
    v
    - (t - r)[:, None] * du_dt
).detach()

meanflow_loss = F.mse_loss(
    u_prediction,
    meanflow_target,
)

print("u prediction:", u_prediction.shape)
print("JVP du/dt:", du_dt.shape)
print("MeanFlow target:", meanflow_target.shape)
print("MeanFlow loss:", meanflow_loss.item())


## 6. MeanFlow reduces to Flow Matching on the diagonal r=t

`r=t`이면 `(t-r)` 항이 0이므로 MeanFlow target은 바로 instantaneous velocity `v`가 된다. 이 관계가 MeanFlow와 Flow Matching의 차이를 가장 직접적으로 보여준다.


In [ ]:
r_equal_t = t.clone()

u_diagonal, du_dt_diagonal = jvp(
    mean_velocity_function,
    (z_t, r_equal_t, t),
    (
        v,
        torch.zeros_like(t),
        torch.ones_like(t),
    ),
)

diagonal_target = (
    v
    - (t - r_equal_t)[:, None] * du_dt_diagonal
)

print(
    "max |MeanFlow target - FM velocity| at r=t:",
    (diagonal_target - v).abs().max().item(),
)


## References and provenance

**DDPM** — Ho et al. epsilon prediction과 forward noising을 참조했다.

**v prediction** — progressive distillation / diffusion parameterization lineage의 `v = alpha*epsilon - sigma*x0`를 사용했다.

**Flow Matching** — Lipman et al. conditional path와 instantaneous vector-field regression을 반영했다.

**Rectified Flow** — Liu et al. straight transport와 reflow viewpoint를 구분해서 적었다.

**MeanFlow** — Geng et al., *Mean Flows for One-step Generative Modeling* (2025) 및 공개 구현. average velocity `u(z_t,r,t)`, MeanFlow identity, tangent `(v,0,1)`에 대한 JVP, stop-gradient target을 반영했다.
